# Golden fixture selection — Denodo → DSI data contract (Work item E)

**Phase 1 (offline, works now):** select the representative `(view_name, db)` pairs from `probe_results.db` and write `fixtures/selection.json`.

Matched to the REAL `probe_results.db` schema:

```
hits(view_name, db, signal_type, property_name, evidence)      -- 11,901 rows
processed(view_name, db, n_signals)                            -- 15,007 rows
```

Because the probe DB stores extracted signals, NOT raw view-details JSON, fixture creation is a two-phase process:

- **PHASE 1 (this notebook):** select the 6 representative pairs + 1 hand-picked, write `fixtures/selection.json`.
- **PHASE 2 (`fetch_fixture_json.ipynb`, online, once the LANL system is back):** call view-details for each selected view, sanitize, and write `fixtures/<label>.json`.

Selection targets (per `DATA_CONTRACT.md` Work item E):

1. `plain` — `n_signals = 0`, not a `vlanl*task` view
2. `doc_url` — description-sourced `etrm.live` URL
3. `access_role` — Details/Access Role Request with `accessit.lanl.gov` URL
4. `ods_link` — ODS Details/ODS DB Link with a real `*_LINK` value
5. `error_500` — one of the 12 `vlanl*task` views (identified as exactly the 12 `vlanl*task` rows with `n_signals=0` — count matches the known error set)
6. `resource_rich` — highest `n_signals` in `processed`
7. `custom_tab` — **HAND-PICKED** (not query-selected): a view confirmed on 2026-07-27 to carry populated non-Summary property-group values in `customTabPropertyMap`, plus the `connectionUris` block. The probe DB cannot select this case by query because Step 0 did not record property-map placement.

In [1]:
import json
import os
import sqlite3

DB_PATH = "probe_results.db"
OUT_DIR = "fixtures"

## Hand-picked fixtures

`custom_tab`: confirmed 2026-07-27 (`test_additional_properties_tab_v10.ipynb`) to carry populated non-Summary group values (`customTabPropertyMap`) and the full `connectionUris` block in its view-details response.

In [2]:
MANUAL_SELECTIONS = {
    "custom_tab": {
        "view_name": "admin_option_type_fvts",
        "db": "dataportal",
        "selection_method": "manual (2026-07-27 Additional-Tab test)",
    },
}

## Selection queries

In [3]:
QUERIES = {
    "plain": (
        "SELECT view_name, db FROM processed "
        "WHERE n_signals = 0 AND view_name NOT LIKE 'vlanl%task%' "
        "ORDER BY view_name LIMIT 1"
    ),
    "doc_url": (
        "SELECT view_name, db FROM hits "
        "WHERE property_name = 'description' AND evidence LIKE '%etrm.live%' "
        "ORDER BY view_name LIMIT 1"
    ),
    "access_role": (
        "SELECT view_name, db FROM hits "
        "WHERE property_name = 'Details/Access Role Request' "
        "AND evidence LIKE '%accessit%' ORDER BY view_name LIMIT 1"
    ),
    "ods_link": (
        "SELECT view_name, db FROM hits "
        "WHERE property_name = 'ODS Details/ODS DB Link' "
        "AND evidence NOT LIKE '%NOLINK%' ORDER BY view_name LIMIT 1"
    ),
    "error_500": (
        "SELECT view_name, db FROM processed "
        "WHERE view_name LIKE 'vlanl%task%' AND n_signals = 0 "
        "ORDER BY view_name LIMIT 1"
    ),
    "resource_rich": (
        "SELECT view_name, db FROM processed "
        "ORDER BY n_signals DESC, view_name LIMIT 1"
    ),
}

## Connect to the probe DB

Run this notebook next to `probe_results.db` (the Step 0 probe output).

In [4]:
assert os.path.exists(DB_PATH), f"{DB_PATH} not found — run next to the Step 0 probe DB."
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row

## Run the query-based selections

In [5]:
selection = {}
print("Selecting fixture views from probe_results.db:")
for label, sql in QUERIES.items():
    row = con.execute(sql).fetchone()
    if row is None:
        print(f"  [{label}] NO MATCH — check query:\n    {sql}")
        continue
    selection[label] = {"view_name": row["view_name"], "db": row["db"]}
    print(f"  [{label}] -> {row['view_name']} ({row['db']})")

Selecting fixture views from probe_results.db:
  [plain] -> access_area_type (dataportal)
  [doc_url] -> ap_1096_data_all (dataportal)
  [access_role] -> affiliation_type_fvts (dataportal)
  [ods_link] -> announcement (dataportal)
  [error_500] -> vlanlchangerequesttask (dataportal)
  [resource_rich] -> ben_prtn_elig_prfl_f (dataportal)


## Merge hand-picked fixtures

In [6]:
for label, sel in MANUAL_SELECTIONS.items():
    selection[label] = dict(sel)
    print(f"  [{label}] -> {sel['view_name']} ({sel['db']})  [manual]")

  [custom_tab] -> admin_option_type_fvts (dataportal)  [manual]


## Attach supporting evidence

Pull the `hits` rows for each selected view — used later in the fixture README.

In [7]:
for label, sel in selection.items():
    ev = con.execute(
        "SELECT signal_type, property_name, evidence FROM hits "
        "WHERE view_name = ? AND db = ?",
        (sel["view_name"], sel["db"]),
    ).fetchall()
    sel["hits"] = [dict(r) for r in ev]
    sel["n_signals"] = len(ev)

# Quick look at the result
{k: {"view_name": v["view_name"], "db": v["db"], "n_signals": v["n_signals"]} for k, v in selection.items()}

{'plain': {'view_name': 'access_area_type',
  'db': 'dataportal',
  'n_signals': 0},
 'doc_url': {'view_name': 'ap_1096_data_all',
  'db': 'dataportal',
  'n_signals': 3},
 'access_role': {'view_name': 'affiliation_type_fvts',
  'db': 'dataportal',
  'n_signals': 4},
 'ods_link': {'view_name': 'announcement', 'db': 'dataportal', 'n_signals': 3},
 'error_500': {'view_name': 'vlanlchangerequesttask',
  'db': 'dataportal',
  'n_signals': 0},
 'resource_rich': {'view_name': 'ben_prtn_elig_prfl_f',
  'db': 'dataportal',
  'n_signals': 6},
 'custom_tab': {'view_name': 'admin_option_type_fvts',
  'db': 'dataportal',
  'n_signals': 4}}

## Write `fixtures/selection.json`

In [8]:
expected_total = len(QUERIES) + len(MANUAL_SELECTIONS)
os.makedirs(OUT_DIR, exist_ok=True)
path = os.path.join(OUT_DIR, "selection.json")
with open(path, "w") as f:
    json.dump(selection, f, indent=2)
print(f"Wrote {path} ({len(selection)}/{expected_total} selections).")
print("Next: once the LANL system is back, run fetch_fixture_json.ipynb "
      "to pull sanitized view-details JSON for these views.")

Wrote fixtures\selection.json (7/7 selections).
Next: once the LANL system is back, run fetch_fixture_json.ipynb to pull sanitized view-details JSON for these views.


In [9]:
selection["custom_tab"]["hits"]

[{'signal_type': 'person_link',
  'property_name': 'Details/Primary Point of Contact',
  'evidence': 'https://pbplus.lanl.gov/search/hbsaenz@lanl.gov'},
 {'signal_type': 'person_link',
  'property_name': 'Details/Data Owner',
  'evidence': 'https://pbplus.lanl.gov/search/txn@lanl.gov'},
 {'signal_type': 'resource_url',
  'property_name': 'Additional Information/More Help for Web Services',
  'evidence': 'https://collaborate.lanl.gov/x/tYV4Cw'},
 {'signal_type': 'resource_url',
  'property_name': 'Additional Information/More Help for Web Services',
  'evidence': 'https://collaborate.lanl.gov/x/tYV4Cw'}]